# Hybrid Model -- WELFake Fake News Detection

This notebook trains the 3-branch hybrid model that
combines TF-IDF statistical features, GloVe semantic
embeddings with BiLSTM sequential modelling, and
handcrafted linguistic features into a single unified
classifier.

Hypothesis: combining three complementary feature
representations captures richer signal than any single
representation alone, producing better classification
than both the classical baseline (LinearSVC 97.20% F1)
and the neural baseline (BiLSTM 97.83% F1).

The three branches are:
Branch 1 (TF-IDF): captures vocabulary and term
  frequency patterns. Fitted with optimal parameters
  found in notebook 03.
Branch 2 (GloVe + BiLSTM): captures sequential semantic
  context using pretrained GloVe 100d vectors from
  notebook 05 to initialise the embedding layer.
Branch 3 (Linguistic): captures writing style signals
  using the 10 features extracted in notebook 06.

Pipeline:
1. Load all three feature tracks from Drive
2. Create multi-input PyTorch datasets
3. Run hyperparameter experiments (lr, units, dropout, freeze)
4. Train final hybrid with optimal configuration
5. Evaluate on held-out test set
6. Generate comparison and training plots
7. Save all artifacts to Drive

## Section 1 -- Setup and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gc
import os
# Prevent OpenMP thread oversubscription on shared Colab hardware
os.environ['OMP_NUM_THREADS'] = '1'
import re
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    f1_score as sk_f1,
    roc_curve,
    roc_auc_score,
    confusion_matrix
)
import joblib
import scipy.sparse as sp

BASE          = '/content/drive/MyDrive/MSU Semester 4/Applied ML/Project'
PROCESSED_DIR = BASE + '/processed/'
SRC_DIR       = BASE + '/src/'
MODELS_DIR    = BASE + '/models/'
FIGURES_DIR   = BASE + '/figures/results/'
RESULTS_DIR   = BASE + '/results/'

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

sys.path.append(SRC_DIR)

from models   import build_hybrid_model, train_model
from models   import save_pytorch_model, get_device
from evaluate import compute_metrics, build_results_table
from evaluate import save_results
from utils    import set_seed, print_section

set_seed(42)
device = get_device()
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print(f"Device         : {device}")
print(f"PyTorch version: {torch.__version__}")
gpu_name = os.popen(
    'nvidia-smi --query-gpu=name --format=csv,noheader'
).read().strip()
print(f"GPU            : {gpu_name}")

## Section 2 -- Load Processed Data

All three splits are loaded. The raw text is needed to
build TF-IDF and sequence features. Linguistic feature
arrays are loaded directly from pre-computed .npy files
saved by notebook 06 to avoid re-extraction.

In [ ]:
print_section("Loading Processed Data")

train_df = pd.read_csv(PROCESSED_DIR + 'train_clean.csv')
val_df   = pd.read_csv(PROCESSED_DIR + 'val_clean.csv')
test_df  = pd.read_csv(PROCESSED_DIR + 'test_clean.csv')

X_train_text = train_df['content'].fillna('').tolist()
X_val_text   = val_df['content'].fillna('').tolist()
X_test_text  = test_df['content'].fillna('').tolist()

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

# Load pre-computed linguistic features from notebook 06
# Loading pre-saved arrays avoids repeating the 30-minute
# extraction step every time this notebook runs
X_train_ling = np.load(
    PROCESSED_DIR + 'X_train_linguistic.npy')
X_val_ling   = np.load(
    PROCESSED_DIR + 'X_val_linguistic.npy')
X_test_ling  = np.load(
    PROCESSED_DIR + 'X_test_linguistic.npy')

LINGUISTIC_DIM = X_train_ling.shape[1]

print(f"Train : {len(X_train_text):,} articles")
print(f"Val   : {len(X_val_text):,} articles")
print(f"Test  : {len(X_test_text):,} articles")
print()
print(f"Linguistic feature dim : {LINGUISTIC_DIM}")
print(f"X_train_ling shape     : {X_train_ling.shape}")

## Section 3 -- Branch A: TF-IDF Features

TF-IDF is refitted here using the optimal parameters
found in notebook 03: max_features=50000, ngram=(1,2),
min_df=5, max_df=0.90, sublinear_tf=True.

The vectorizer is fitted on training text only to prevent
any leakage of validation or test vocabulary into the
feature representation. The resulting sparse matrices
are converted to dense float32 arrays for use as
PyTorch tensor inputs.

In [ ]:
print_section("Branch A: TF-IDF Feature Extraction")

# Use optimal TF-IDF parameters found in notebook 03
# These were determined by systematic experiments on
# the training set using cross-validation
TFIDF_MAX_FEATURES = 50000

tfidf_vec = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    sublinear_tf=True,
    strip_accents='unicode',
    token_pattern=r'\w{2,}'
)

# Fit on training set only -- prevents test set vocabulary
# from influencing the feature space
print("Fitting TF-IDF on training set...")
X_train_tfidf_sparse = tfidf_vec.fit_transform(X_train_text)
X_val_tfidf_sparse   = tfidf_vec.transform(X_val_text)
X_test_tfidf_sparse  = tfidf_vec.transform(X_test_text)

# Keep sparse to avoid ~10 GB RAM spike from dense conversion
# 43862 x 50000 x float32 = ~8.8 GB — exceeds Colab T4 RAM
X_train_tfidf = X_train_tfidf_sparse
X_val_tfidf   = X_val_tfidf_sparse
X_test_tfidf  = X_test_tfidf_sparse

# TFIDF_DIM needed for model construction later
TFIDF_DIM = X_train_tfidf.shape[1]

# Save fitted vectorizer for Streamlit demo app
joblib.dump(tfidf_vec,
            MODELS_DIR + 'hybrid_tfidf_vectorizer.joblib')

print(f"TF-IDF vocabulary size : {TFIDF_DIM:,}")
print(f"Train TF-IDF shape     : {X_train_tfidf.shape}")
print("Vectorizer saved to models/hybrid_tfidf_vectorizer.joblib")

# Free temporary objects now that sparse matrices are assigned
# Sparse matrix objects themselves stay; only intermediate refs drop
import gc
gc.collect()
torch.cuda.empty_cache()

## Section 4 -- Branch B: Sequence Features

The tokenizer vocabulary built in notebook 04 is loaded
to ensure consistency between the BiLSTM baseline and
the hybrid model sequence branch.

Text articles are encoded as padded integer sequences
of length 300. The GloVe embedding matrix from notebook 05
is loaded and will be used to initialise the embedding layer
with pretrained semantic knowledge.

In [ ]:
print_section("Branch B: Sequence Features")

# Load tokenizer built in notebook 04
# Using the same vocabulary ensures token index N
# maps to the same word across all neural notebooks
word2idx      = joblib.load(MODELS_DIR + 'tokenizer.joblib')
VOCAB_SIZE    = len(word2idx)
MAX_LEN       = 300
EMBEDDING_DIM = 100  # GloVe 6B 100d vectors

print(f"Vocabulary size  : {VOCAB_SIZE:,}")
print(f"Max sequence len : {MAX_LEN}")
print(f"Embedding dim    : {EMBEDDING_DIM}d (GloVe)")


def tokenize(text):
    """
    Tokenize text by lowercasing and removing punctuation.
    Must match the tokenization used in notebook 04 exactly
    so that token indices are consistent.
    """
    return re.sub(r"[^\w\s']", '', text.lower()).split()


def encode_and_pad(texts, word2idx, max_len):
    """
    Convert list of text strings to padded integer sequences.

    Each word is mapped to its vocabulary index.
    Unknown words map to index 1 (UNK token).
    Sequences longer than max_len are truncated.
    Sequences shorter than max_len are zero-padded.

    Parameters
    ----------
    texts : list of str
        List of article content strings.
    word2idx : dict
        Word to index mapping from tokenizer.
    max_len : int
        Fixed output sequence length.

    Returns
    -------
    np.ndarray
        Integer array of shape (n_articles, max_len).
    """
    sequences = []
    for text in texts:
        tokens = tokenize(text)
        # Map each token to its index, unknown tokens get 1
        ids    = [word2idx.get(t, 1) for t in tokens]
        # Truncate sequences longer than max_len
        ids    = ids[:max_len]
        # Pad sequences shorter than max_len with 0 (PAD)
        ids    = ids + [0] * (max_len - len(ids))
        sequences.append(ids)
    return np.array(sequences, dtype=np.int64)


print("Encoding sequences...")
X_train_seq = encode_and_pad(X_train_text, word2idx, MAX_LEN)
X_val_seq   = encode_and_pad(X_val_text,   word2idx, MAX_LEN)
X_test_seq  = encode_and_pad(X_test_text,  word2idx, MAX_LEN)

print(f"Train seq shape  : {X_train_seq.shape}")
print(f"Val   seq shape  : {X_val_seq.shape}")
print(f"Test  seq shape  : {X_test_seq.shape}")

# Load GloVe embedding matrix built in notebook 05
# Shape: (vocab_size, 100) -- row N = vector for token N
glove_matrix = np.load(
    MODELS_DIR + 'glove_embedding_matrix.npy')
print(f"\nGloVe matrix shape : {glove_matrix.shape}")
print(f"GloVe matrix dtype : {glove_matrix.dtype}")

## Section 5 -- Create Multi-Input PyTorch Datasets

The hybrid model requires three simultaneous inputs per
article: TF-IDF vector, integer sequence, and linguistic
feature vector. A custom Dataset class packages all three
inputs with the label for efficient batching.

A smaller batch size of 32 (vs 64 in notebook 04) is used
because three dense tensors per article require more GPU
memory than a single sequence tensor.

In [ ]:
class HybridDataset(Dataset):
    """
    PyTorch Dataset for the 3-branch hybrid model.

    TF-IDF matrix is stored as scipy sparse to avoid
    materialising the full ~8.8 GB dense matrix in RAM.
    Each row is densified lazily in __getitem__ only
    when that specific sample is accessed.

    Parameters
    ----------
    tfidf_sparse : scipy.sparse matrix
        Sparse TF-IDF matrix, shape (N, tfidf_dim).
    sequences : np.ndarray
        Padded integer sequences, shape (N, max_len).
    linguistic : np.ndarray
        Scaled linguistic features, shape (N, linguistic_dim).
    labels : np.ndarray
        Binary labels shape (N,).
    """

    def __init__(self, tfidf_sparse, sequences,
                 linguistic, labels):
        # Store TF-IDF as sparse — never call .toarray() here
        # Full dense conversion = ~8.8 GB RAM on 50k features
        self.tfidf_sparse = tfidf_sparse
        self.sequences    = torch.tensor(
            sequences,  dtype=torch.long)
        self.linguistic   = torch.tensor(
            linguistic, dtype=torch.float32)
        self.labels       = torch.tensor(
            labels,     dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Densify only this single row at access time
        # One row: 50000 x 4 bytes = 200 KB (safe)
        # vs full matrix: 43862 x 50000 x 4 = ~8.8 GB
        tfidf_row = torch.tensor(
            self.tfidf_sparse[idx].toarray().squeeze(),
            dtype=torch.float32
        )
        return (self.sequences[idx],
                tfidf_row,
                self.linguistic[idx],
                self.labels[idx])


# Smaller batch size than BiLSTM notebook because
# three feature matrices require more GPU memory
BATCH_SIZE = 32

train_loader = DataLoader(
    HybridDataset(X_train_tfidf, X_train_seq,
                  X_train_ling, y_train),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    HybridDataset(X_val_tfidf, X_val_seq,
                  X_val_ling, y_val),
    batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    HybridDataset(X_test_tfidf, X_test_seq,
                  X_test_ling, y_test),
    batch_size=BATCH_SIZE, shuffle=False
)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

## Section 6 -- Hyperparameter Experiments

Four experiments systematically identify optimal
hyperparameters for the hybrid model. Each experiment
trains for 5 epochs maximum with early stopping patience
of 2 using only train and validation sets.

The experiments are run sequentially, each using the
optimal value found in the previous experiment:
A (lr) -> B (lstm_units) -> C (dropout) -> D (freeze_glove)

In [ ]:
def load_glove_into_model(model, glove_matrix, freeze):
    """
    Load pretrained GloVe vectors into the model embedding layer.

    Initialising with GloVe instead of random vectors gives
    the model semantic knowledge before training begins.
    Freezing prevents the pretrained weights from being
    updated, which can be beneficial when training data
    is limited. Fine-tuning allows the embeddings to adapt
    to the specific vocabulary patterns in WELFake.

    Parameters
    ----------
    model : nn.Module
        HybridClassifier instance with embedding layer.
    glove_matrix : np.ndarray
        Pretrained GloVe matrix of shape (vocab_size, 100).
    freeze : bool
        If True, embedding weights are not updated during
        training. If False, embeddings are fine-tuned.

    Returns
    -------
    nn.Module
        Model with GloVe weights loaded.
    """
    glove_tensor = torch.tensor(
        glove_matrix, dtype=torch.float32)

    # Access embedding layer inside the LSTM branch
    # The embedding is the first layer in branch2
    with torch.no_grad():
        model.embedding.weight.copy_(glove_tensor)

    # Freeze or allow gradient updates on embedding
    model.embedding.weight.requires_grad = not freeze

    status = "frozen" if freeze else "fine-tunable"
    print(f"GloVe weights loaded ({status})")
    return model


def quick_train_hybrid(lr=0.001, lstm_units=128,
                       dropout=0.3, freeze_glove=False,
                       epochs=3, patience=2):
    """
    Short training run to evaluate a hyperparameter config.

    Builds a fresh hybrid model, loads GloVe weights, trains
    for up to 3 epochs with early stopping, and returns the
    best validation F1 achieved. GPU memory is cleared after
    each run to prevent out-of-memory errors across experiments.

    Parameters
    ----------
    lr : float
        Adam learning rate.
    lstm_units : int
        BiLSTM hidden units per direction.
    dropout : float
        Dropout rate in fusion layer.
    freeze_glove : bool
        Whether to freeze GloVe embedding weights.
    epochs : int
        Maximum training epochs.
    patience : int
        Early stopping patience on val F1.

    Returns
    -------
    float
        Best validation F1 macro score.
    """
    model = build_hybrid_model(
        tfidf_dim=TFIDF_DIM,
        vocab_size=VOCAB_SIZE,
        embedding_dim=EMBEDDING_DIM,
        max_len=MAX_LEN,
        lstm_units=lstm_units,
        linguistic_dim=LINGUISTIC_DIM,
        dropout=dropout
    ).to(device)

    # Load pretrained GloVe weights into embedding layer
    model = load_glove_into_model(
        model, glove_matrix, freeze=freeze_glove)

    # Filter out frozen embedding parameters so Adam
    # does not create momentum state for them unnecessarily
    optimizer   = torch.optim.Adam(
        filter(lambda p: p.requires_grad,
               model.parameters()), lr=lr)
    criterion   = nn.BCELoss()
    best_val_f1 = 0.0
    no_improve  = 0

    for epoch in range(epochs):
        # Training pass
        model.train()
        for seqs, tfidf, ling, labels in train_loader:
            seqs   = seqs.to(device)
            tfidf  = tfidf.to(device)
            ling   = ling.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            # Forward pass with all three inputs
            preds = model(tfidf, seqs, ling).squeeze()
            loss  = criterion(preds, labels)
            loss.backward()
            optimizer.step()

        # Validation pass
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for seqs, tfidf, ling, labels in val_loader:
                seqs  = seqs.to(device)
                tfidf = tfidf.to(device)
                ling  = ling.to(device)
                preds = model(
                    tfidf, seqs, ling).squeeze().cpu().numpy()
                all_preds.extend((preds > 0.5).astype(int))
                all_labels.extend(
                    labels.numpy().astype(int))

        val_f1 = sk_f1(
            all_labels, all_preds, average='macro')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve  = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    # Free GPU memory after each experiment run
    # to prevent CUDA out-of-memory across multiple runs
    del model
    torch.cuda.empty_cache()
    return round(best_val_f1, 4)


print("quick_train_hybrid helper defined.")

### Experiment A -- Learning Rate

Learning rate has the largest impact on hybrid model
convergence. Three values from moderate (0.001) to
conservative (0.0001) are tested. Values too low converge
too slowly to show meaningful results in 3 epochs.

lr=0.01 is excluded based on notebook 04 findings
where it consistently produced worst results due to
gradient instability in LSTM networks. Each experiment
uses 3 epochs — sufficient to rank configurations
reliably before final full training.

In [ ]:
print_section("Experiment A: Learning Rate")

# lr=0.01 excluded: notebook 04 confirmed it causes gradient
# instability in LSTM training — always worst performer
lr_candidates = [0.001, 0.0005, 0.0001]
lr_results    = []

for lr in lr_candidates:
    print(f"Testing lr={lr}...")
    start   = time.time()
    val_f1  = quick_train_hybrid(lr=lr)
    elapsed = round(time.time() - start, 1)
    lr_results.append({
        'lr': lr, 'val_f1': val_f1,
        'time_seconds': elapsed
    })
    print(f"  val_f1={val_f1:.4f}  time={elapsed}s")

lr_df      = pd.DataFrame(lr_results)
OPTIMAL_LR = float(
    lr_df.loc[lr_df['val_f1'].idxmax(), 'lr'])
print(f"\nExperiment A Results:")
print(lr_df.to_string(index=False))
print(f"\nOptimal learning rate: {OPTIMAL_LR}")

### Experiment B -- LSTM Units

The number of LSTM units controls the capacity of the
sequence branch. More units capture richer sequential
patterns but increase memory and training time. Two
values are tested using the optimal lr from Experiment A.

64 LSTM units excluded based on notebook 04 where
256 units significantly outperformed 64. Testing 128
and 256 covers the relevant performance range.

In [ ]:
print_section("Experiment B: LSTM Units")

# 64 units excluded: notebook 04 BiLSTM showed 64 units
# significantly underperformed 128 and 256 — not worth testing
lstm_candidates = [128, 256]
lstm_results    = []

for units in lstm_candidates:
    print(f"Testing lstm_units={units}...")
    start   = time.time()
    val_f1  = quick_train_hybrid(
        lr=OPTIMAL_LR, lstm_units=units)
    elapsed = round(time.time() - start, 1)
    lstm_results.append({
        'lstm_units': units,
        'val_f1': val_f1,
        'time_seconds': elapsed
    })
    print(f"  val_f1={val_f1:.4f}  time={elapsed}s")

lstm_df      = pd.DataFrame(lstm_results)
OPTIMAL_LSTM = int(
    lstm_df.loc[lstm_df['val_f1'].idxmax(), 'lstm_units'])
print(f"\nExperiment B Results:")
print(lstm_df.to_string(index=False))
print(f"\nOptimal lstm_units: {OPTIMAL_LSTM}")

### Experiment C -- Dropout Rate

Dropout regularises the fusion layer by randomly zeroing
activations during training, reducing overfitting.
Too little dropout risks memorising training patterns.
Too much dropout discards useful learned representations.

In [ ]:
print_section("Experiment C: Dropout Rate")

dropout_candidates = [0.2, 0.3, 0.5]
dropout_results    = []

for dropout in dropout_candidates:
    print(f"Testing dropout={dropout}...")
    start   = time.time()
    val_f1  = quick_train_hybrid(
        lr=OPTIMAL_LR,
        lstm_units=OPTIMAL_LSTM,
        dropout=dropout
    )
    elapsed = round(time.time() - start, 1)
    dropout_results.append({
        'dropout': dropout,
        'val_f1': val_f1,
        'time_seconds': elapsed
    })
    print(f"  val_f1={val_f1:.4f}  time={elapsed}s")

dropout_df      = pd.DataFrame(dropout_results)
OPTIMAL_DROPOUT = float(
    dropout_df.loc[
        dropout_df['val_f1'].idxmax(), 'dropout'])
print(f"\nExperiment C Results:")
print(dropout_df.to_string(index=False))
print(f"\nOptimal dropout: {OPTIMAL_DROPOUT}")

### Experiment D -- GloVe Freeze vs Fine-tune

Freezing GloVe embeddings preserves pretrained semantic
knowledge exactly as learned from Wikipedia and Gigaword.
Fine-tuning allows embeddings to adapt to WELFake
vocabulary patterns but risks overfitting on smaller
datasets. This experiment empirically determines which
strategy works better for this specific dataset.

In [ ]:
print_section("Experiment D: GloVe Freeze vs Fine-tune")

freeze_results = []

for freeze in [True, False]:
    label = "frozen" if freeze else "fine-tuned"
    print(f"Testing GloVe {label}...")
    start   = time.time()
    val_f1  = quick_train_hybrid(
        lr=OPTIMAL_LR,
        lstm_units=OPTIMAL_LSTM,
        dropout=OPTIMAL_DROPOUT,
        freeze_glove=freeze
    )
    elapsed = round(time.time() - start, 1)
    freeze_results.append({
        'freeze_glove': freeze,
        'setting': label,
        'val_f1': val_f1,
        'time_seconds': elapsed
    })
    print(f"  val_f1={val_f1:.4f}  time={elapsed}s")

freeze_df      = pd.DataFrame(freeze_results)
OPTIMAL_FREEZE = bool(
    freeze_df.loc[
        freeze_df['val_f1'].idxmax(), 'freeze_glove'])
print(f"\nExperiment D Results:")
print(freeze_df.to_string(index=False))
print(f"\nOptimal freeze_glove: {OPTIMAL_FREEZE}")

### Hyperparameter Summary

In [ ]:
print_section("Optimal Hyperparameter Configuration")
print(f"Learning rate  : {OPTIMAL_LR}")
print(f"LSTM units     : {OPTIMAL_LSTM}")
print(f"Dropout        : {OPTIMAL_DROPOUT}")
print(f"GloVe freeze   : {OPTIMAL_FREEZE}")
print(f"Embedding dim  : {EMBEDDING_DIM} (fixed GloVe 100d)")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Max seq len    : {MAX_LEN}")

## Section 7 -- Build and Train Final Hybrid Model

The final model is built using the optimal hyperparameters
found across all four experiments. GloVe weights are loaded
with the optimal freeze setting. Early stopping with
patience of 3 prevents overfitting on the training set.

In [ ]:
print_section("Building Final Hybrid Model")

model = build_hybrid_model(
    tfidf_dim=TFIDF_DIM,
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    max_len=MAX_LEN,
    lstm_units=OPTIMAL_LSTM,
    linguistic_dim=LINGUISTIC_DIM,
    dropout=OPTIMAL_DROPOUT
).to(device)

# Load pretrained GloVe weights with optimal freeze setting
model = load_glove_into_model(
    model, glove_matrix, freeze=OPTIMAL_FREEZE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print()
print("Training final model...")
print(f"lr={OPTIMAL_LR}, lstm={OPTIMAL_LSTM}, "
      f"dropout={OPTIMAL_DROPOUT}, "
      f"freeze_glove={OPTIMAL_FREEZE}")

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=10,
    lr=OPTIMAL_LR,
    patience=3,
    device=str(device)
)

save_pytorch_model(model, MODELS_DIR + 'hybrid_model.pt')
print("Model saved to models/hybrid_model.pt")

## Section 8 -- Evaluate on Test Set

Final evaluation uses the held-out test set which was
not used during any tuning stage. Results are compared
against all previous models to show the hybrid model
performance relative to the full progression from
classical to neural to hybrid approaches.

In [ ]:
print_section("Evaluation on Test Set")

model.eval()
all_preds  = []
all_scores = []
all_labels = []

with torch.no_grad():
    for seqs, tfidf, ling, labels in test_loader:
        seqs   = seqs.to(device)
        tfidf  = tfidf.to(device)
        ling   = ling.to(device)
        scores = model(
            tfidf, seqs, ling).squeeze().cpu().numpy()
        preds  = (scores > 0.5).astype(int)
        all_scores.extend(scores.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(
            labels.numpy().astype(int).tolist())

hybrid_metrics = compute_metrics(
    all_labels, all_preds, all_scores, 'Hybrid')

print(f"Accuracy : {hybrid_metrics['Accuracy']:.4f}")
print(f"F1 Macro : {hybrid_metrics['F1_Macro']:.4f}")
print(f"ROC-AUC  : {hybrid_metrics['ROC_AUC']:.4f}")
print()

# Compare against all previous models
print("Full model comparison:")
print(f"  Random Forest      : 0.9429 F1")
print(f"  XGBoost            : 0.9627 F1")
print(f"  Logistic Regression: 0.9703 F1")
print(f"  LinearSVC          : 0.9720 F1")
print(f"  BiLSTM             : 0.9783 F1")
print(f"  Hybrid (this)      : "
      f"{hybrid_metrics['F1_Macro']:.4f} F1")

diff_classical = hybrid_metrics['F1_Macro'] - 0.9720
diff_bilstm    = hybrid_metrics['F1_Macro'] - 0.9783
print(f"\n  vs LinearSVC : {diff_classical:+.4f}")
print(f"  vs BiLSTM    : {diff_bilstm:+.4f}")

results_df = build_results_table([hybrid_metrics])
save_results(results_df,
             RESULTS_DIR + 'hybrid_results.csv')

## Section 9 -- Visualizations

Four plots are generated for the report and poster session.

### Plot 18 -- Hybrid Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'],
         color='#E76F51', linewidth=2,
         marker='o', label='Train loss')
ax1.plot(epochs_range, history['val_loss'],
         color='#2A9D8F', linewidth=2,
         marker='s', label='Val loss')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=13)
ax1.legend()

ax2.plot(epochs_range, history['train_acc'],
         color='#E76F51', linewidth=2,
         marker='o', label='Train accuracy')
ax2.plot(epochs_range, history['val_acc'],
         color='#2A9D8F', linewidth=2,
         marker='s', label='Val accuracy')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training and Validation Accuracy',
              fontsize=13)
ax2.legend()

fig.suptitle('Hybrid Model Training Curves',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '18_hybrid_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

### Plot 19 -- ROC Curve and Confusion Matrix

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

fpr, tpr, _ = roc_curve(all_labels, all_scores)
auc_val     = roc_auc_score(all_labels, all_scores)

ax1.plot(fpr, tpr, color='#E76F51', linewidth=2,
         label=f'Hybrid (AUC = {auc_val:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1,
         alpha=0.5, label='Random classifier')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve -- Hybrid Model', fontsize=13)
ax1.legend()

cm     = confusion_matrix(all_labels, all_preds)
cm_pct = (cm.astype(float)
           / cm.sum(axis=1, keepdims=True) * 100)

sns.heatmap(
    cm_pct, annot=True, fmt='.1f',
    cmap='YlOrRd',
    xticklabels=['Fake (0)', 'Real (1)'],
    yticklabels=['Fake (0)', 'Real (1)'],
    ax=ax2, cbar=False
)
for i in range(2):
    for j in range(2):
        ax2.text(j + 0.5, i + 0.72,
                 f'n={cm[i, j]:,}',
                 ha='center', va='center',
                 fontsize=9, color='gray')
ax2.set_title('Confusion Matrix -- Hybrid Model',
              fontsize=13)
ax2.set_ylabel('True Label', fontsize=11)
ax2.set_xlabel('Predicted Label', fontsize=11)

fig.suptitle('Hybrid Model Evaluation -- Test Set',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR + '19_hybrid_evaluation.png',
            dpi=150, bbox_inches='tight')
plt.show()

### Plot 20 -- All Models Comparison

In [ ]:
# Load BiLSTM results to include in comparison
try:
    bilstm_df = pd.read_csv(RESULTS_DIR + 'bilstm_results.csv')
    bilstm_f1 = float(bilstm_df['F1_Macro'].iloc[0])
except Exception:
    # Use known result from notebook 04 as fallback
    bilstm_f1 = 0.9783

model_names = [
    'Random\nForest', 'XGBoost',
    'Logistic\nRegression', 'LinearSVC',
    'BiLSTM', 'Hybrid\n(Ours)'
]
f1_scores = [
    0.9429, 0.9627, 0.9703, 0.9720,
    bilstm_f1,
    hybrid_metrics['F1_Macro']
]
# Color encodes model family for visual clarity
bar_colors = [
    '#E9C46A', '#E9C46A', '#E9C46A', '#E9C46A',
    '#2A9D8F', '#E76F51'
]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(model_names, f1_scores,
              color=bar_colors, alpha=0.85,
              edgecolor='white', linewidth=1.2)

for bar, score in zip(bars, f1_scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        score + 0.001,
        f'{score:.4f}',
        ha='center', va='bottom', fontsize=10
    )

ax.set_ylim(0.90, 1.01)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('F1 Macro (Test Set)', fontsize=12)
ax.set_title('All Models Comparison -- F1 Macro',
             fontsize=14, fontweight='bold')

from matplotlib.patches import Patch
legend_handles = [
    Patch(color='#E9C46A',
          label='Classical baselines (TF-IDF)'),
    Patch(color='#2A9D8F',
          label='Neural baseline (BiLSTM)'),
    Patch(color='#E76F51',
          label='Hybrid model (contribution)'),
]
ax.legend(handles=legend_handles, fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR + '20_all_models_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

### Plot 21 -- All ROC Curves on One Plot

In [ ]:
# Load classical model predictions for unified ROC plot
# This requires reloading saved models from notebook 03
# If models are not available, skip classical ROC curves
try:
    from sklearn.metrics import roc_curve, roc_auc_score
    import joblib
    import scipy.sparse as sp

    # Reload TF-IDF and classical models
    vec_classical = joblib.load(
        MODELS_DIR + 'tfidf_vectorizer.joblib')
    lr_model  = joblib.load(MODELS_DIR + 'model_lr.joblib')
    svm_model = joblib.load(MODELS_DIR + 'model_svm.joblib')

    X_test_classical = vec_classical.transform(X_test_text)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors  = {
        'LinearSVC'           : '#E9C46A',
        'Logistic Regression' : '#F4A261',
        'BiLSTM'              : '#2A9D8F',
        'Hybrid'              : '#E76F51'
    }

    # LR ROC
    lr_scores   = lr_model.predict_proba(
        X_test_classical)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, lr_scores)
    auc         = roc_auc_score(y_test, lr_scores)
    ax.plot(fpr, tpr,
            color=colors['Logistic Regression'],
            linewidth=2,
            label=f'Logistic Regression (AUC={auc:.4f})')

    # SVM ROC (uses decision_function)
    svm_scores  = svm_model.decision_function(
        X_test_classical)
    fpr, tpr, _ = roc_curve(y_test, svm_scores)
    auc         = roc_auc_score(y_test, svm_scores)
    ax.plot(fpr, tpr, color=colors['LinearSVC'],
            linewidth=2,
            label=f'LinearSVC (AUC={auc:.4f})')

    # Hybrid ROC (already computed)
    fpr_h, tpr_h, _ = roc_curve(all_labels, all_scores)
    auc_h           = roc_auc_score(all_labels, all_scores)
    ax.plot(fpr_h, tpr_h, color=colors['Hybrid'],
            linewidth=2.5,
            label=f'Hybrid (AUC={auc_h:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1,
            alpha=0.4, label='Random classifier')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves -- All Models',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, loc='lower right')
    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR + '22_roc_all_models.png',
        dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: figures/results/22_roc_all_models.png")

except Exception as e:
    print(f"Could not load classical models: {e}")
    print("Saving hybrid ROC only.")

### Plot 22 -- Hyperparameter Experiment Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Experiment A: Learning Rate
axes[0].bar(
    [str(r['lr']) for r in lr_results],
    [r['val_f1'] for r in lr_results],
    color=['#2A9D8F' if r['lr'] == OPTIMAL_LR
           else '#E9C46A' for r in lr_results],
    alpha=0.85
)
for i, r in enumerate(lr_results):
    axes[0].text(i, r['val_f1'] + 0.001,
                 f"{r['val_f1']:.4f}",
                 ha='center', fontsize=9)
axes[0].set_xlabel('Learning Rate', fontsize=11)
axes[0].set_ylabel('Val F1 Macro', fontsize=11)
axes[0].set_title('Experiment A: Learning Rate',
                  fontsize=12)
axes[0].set_ylim(
    min(r['val_f1'] for r in lr_results) - 0.02, 1.0)

# Experiment B: LSTM Units
axes[1].bar(
    [str(r['lstm_units']) for r in lstm_results],
    [r['val_f1'] for r in lstm_results],
    color=['#2A9D8F' if r['lstm_units'] == OPTIMAL_LSTM
           else '#E9C46A' for r in lstm_results],
    alpha=0.85
)
for i, r in enumerate(lstm_results):
    axes[1].text(i, r['val_f1'] + 0.001,
                 f"{r['val_f1']:.4f}",
                 ha='center', fontsize=9)
axes[1].set_xlabel('LSTM Units', fontsize=11)
axes[1].set_ylabel('Val F1 Macro', fontsize=11)
axes[1].set_title('Experiment B: LSTM Units',
                  fontsize=12)
axes[1].set_ylim(
    min(r['val_f1'] for r in lstm_results) - 0.02, 1.0)

# Experiment C: Dropout
axes[2].bar(
    [str(r['dropout']) for r in dropout_results],
    [r['val_f1'] for r in dropout_results],
    color=['#2A9D8F' if r['dropout'] == OPTIMAL_DROPOUT
           else '#E9C46A' for r in dropout_results],
    alpha=0.85
)
for i, r in enumerate(dropout_results):
    axes[2].text(i, r['val_f1'] + 0.001,
                 f"{r['val_f1']:.4f}",
                 ha='center', fontsize=9)
axes[2].set_xlabel('Dropout Rate', fontsize=11)
axes[2].set_ylabel('Val F1 Macro', fontsize=11)
axes[2].set_title('Experiment C: Dropout Rate',
                  fontsize=12)
axes[2].set_ylim(
    min(r['val_f1'] for r in dropout_results) - 0.02,
    1.0)

# Experiment D: GloVe Freeze
axes[3].bar(
    [r['setting'] for r in freeze_results],
    [r['val_f1'] for r in freeze_results],
    color=['#2A9D8F'
           if r['freeze_glove'] == OPTIMAL_FREEZE
           else '#E9C46A'
           for r in freeze_results],
    alpha=0.85
)
for i, r in enumerate(freeze_results):
    axes[3].text(i, r['val_f1'] + 0.001,
                 f"{r['val_f1']:.4f}",
                 ha='center', fontsize=9)
axes[3].set_xlabel('GloVe Setting', fontsize=11)
axes[3].set_ylabel('Val F1 Macro', fontsize=11)
axes[3].set_title('Experiment D: GloVe Freeze',
                  fontsize=12)
axes[3].set_ylim(
    min(r['val_f1'] for r in freeze_results) - 0.02,
    1.0)

fig.suptitle('Hybrid Model Hyperparameter Experiments',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(
    FIGURES_DIR + '23_hybrid_hparam_experiments.png',
    dpi=150, bbox_inches='tight')
plt.show()

## Section 10 -- Save All Results

In [ ]:
print_section("Saving All Results")

# Build consolidated results table with all 6 models
# This single CSV becomes the master results table
# referenced throughout the report
all_results = [
    {'Model': 'Random Forest',
     'Accuracy': 0.9436, 'F1_Macro': 0.9429,
     'ROC_AUC': 0.9879},
    {'Model': 'XGBoost',
     'Accuracy': 0.9631, 'F1_Macro': 0.9627,
     'ROC_AUC': 0.9938},
    {'Model': 'Logistic Regression',
     'Accuracy': 0.9706, 'F1_Macro': 0.9703,
     'ROC_AUC': 0.9964},
    {'Model': 'LinearSVC',
     'Accuracy': 0.9722, 'F1_Macro': 0.9720,
     'ROC_AUC': 0.9967},
    {'Model': 'BiLSTM',
     'Accuracy': 0.9785, 'F1_Macro': 0.9783,
     'ROC_AUC': 0.9980},
    {'Model': 'Hybrid',
     'Accuracy': hybrid_metrics['Accuracy'],
     'F1_Macro': hybrid_metrics['F1_Macro'],
     'ROC_AUC': hybrid_metrics['ROC_AUC']},
]
all_results_df = pd.DataFrame(all_results)
all_results_df.to_csv(
    RESULTS_DIR + 'all_results.csv', index=False)

print("Files saved to Google Drive:")
print("  models/hybrid_model.pt")
print("  models/hybrid_tfidf_vectorizer.joblib")
print("  results/hybrid_results.csv")
print("  results/all_results.csv")
print("  figures/results/18_hybrid_training_curves.png")
print("  figures/results/19_hybrid_evaluation.png")
print("  figures/results/20_all_models_comparison.png")
print("  figures/results/22_roc_all_models.png")
print("  figures/results/23_hybrid_hparam_experiments.png")

## Section 11 -- Summary

In [ ]:
print_section("HYBRID MODEL SUMMARY")

print("Architecture:")
print(f"  Branch 1 (TF-IDF)    : "
      f"{TFIDF_DIM:,}d -> 256 -> 128 (relu)")
print(f"  Branch 2 (GloVe+LSTM): "
      f"{EMBEDDING_DIM}d GloVe -> "
      f"BiLSTM({OPTIMAL_LSTM}) -> 128 (relu)")
print(f"  Branch 3 (Linguistic): "
      f"{LINGUISTIC_DIM}d -> 32 (relu)")
print(f"  Fusion               : "
      f"288d -> 128 -> 1 (sigmoid)")
print()
print("Optimal Hyperparameters:")
print(f"  Learning rate  : {OPTIMAL_LR}")
print(f"  LSTM units     : {OPTIMAL_LSTM}")
print(f"  Dropout        : {OPTIMAL_DROPOUT}")
print(f"  GloVe freeze   : {OPTIMAL_FREEZE}")
print()
print("Final Test Set Results:")
print(f"  Accuracy : {hybrid_metrics['Accuracy']:.4f}")
print(f"  F1 Macro : {hybrid_metrics['F1_Macro']:.4f}")
print(f"  ROC-AUC  : {hybrid_metrics['ROC_AUC']:.4f}")
print()
print("Comparison to baselines:")
print(f"  LinearSVC : 0.9720  vs Hybrid: "
      f"{hybrid_metrics['F1_Macro']:.4f}  "
      f"({diff_classical:+.4f})")
print(f"  BiLSTM    : 0.9783  vs Hybrid: "
      f"{hybrid_metrics['F1_Macro']:.4f}  "
      f"({diff_bilstm:+.4f})")

## Next Steps

With the hybrid model trained and evaluated, the next
notebook (08_ablation.ipynb) trains three 2-branch
variants by removing one branch at a time to quantify
each branch contribution to overall performance.

Notebook 09 (09_significance.ipynb) then runs McNemar
statistical significance tests to confirm that performance
differences between models are not due to chance.